# The 7 Fleet Primitives Wrapping the ADK Agent Tree

**What this notebook shows**: the 7 Fleet primitives from `gemini_hackathon/agents/fleet/` (the byte-identical wholesale copy of `cianfhoghlaim/packages/fleet/src/cianfhoghlaim/fleet/`).

**Source**: `gemini_hackathon/agents/fleet/{fleet_gateway,fleet_identity,fleet_model_armor,fleet_memory,fleet_observability,fleet_mcp_curriculum,fleet_agui}.py` — 3,444 LOC total.


## The 7 Fleet primitives

1. **FleetGateway** — single canonical entrypoint + keyword routing (`AGENT_NAMES` / `KEYWORD_TO_AGENT`)
2. **FleetIdentity** — caller identity resolution (BetterAuth / JWT / anonymous)
3. **FleetModelArmor** — input sanitisation (PII redaction, prompt-injection guard, jailbreak detection)
4. **FleetMemory** — Letta-backed long-term memory (with in-memory fallback)
5. **FleetObservability** — Langfuse + Logfire + MLflow + GCP-native OTel tracing
6. **FleetMcpCurriculum** — the 14-subject MCP server (the ADK tools surface as MCP resources)
7. **FleetAGUIBridge** — the CopilotKit + AGUI protocol adapter


In [ ]:
from gemini_hackathon.agents.fleet import (
    FleetAGUIBridge,
    FleetGateway,
    FleetIdentity,
    FleetMcpCurriculum,
    FleetMemory,
    FleetModelArmor,
    FleetObservability,
)

primitives = [
    ("FleetGateway", FleetGateway, "Single canonical entrypoint + keyword routing"),
    ("FleetIdentity", FleetIdentity, "Caller identity resolution (BetterAuth / JWT)"),
    ("FleetModelArmor", FleetModelArmor, "Input sanitisation (PII / prompt-injection)"),
    ("FleetMemory", FleetMemory, "Letta-backed long-term memory"),
    ("FleetObservability", FleetObservability, "Langfuse + Logfire + MLflow + OTel"),
    ("FleetMcpCurriculum", FleetMcpCurriculum, "14-subject MCP server"),
    ("FleetAGUIBridge", FleetAGUIBridge, "CopilotKit + AGUI protocol adapter"),
]
for name, _cls, doc in primitives:
    print(f"  - {name}: {doc}")

## How the primitives wrap `run_agent_turn()`

Per `gemini_hackathon/agents/adk_gemini_agent.py:run_agent_turn()`:
1. `ModelArmor.check_prompt(message)` (Fleet primitive #3) — blocks prompt injection + PII
2. `Observability.trace(agent_name, user_id, session_id, subnation)` (Fleet primitive #5) — opens a trace
3. `runner.run(user_id, session_id, new_message=content)` — the real ADK invocation
4. `Observability.record_invocation(trace, ...)` (Fleet primitive #5) — emits the cost + tokens event
5. `render_agui_events(raw_events)` → `AgUiEvent` for the `FleetAGUIBridge` (Fleet primitive #7) to surface to CopilotKit

Identity is provided by `FleetIdentity.authenticate()` (Fleet primitive #2) and the long-term memory by `FleetMemory` (Fleet primitive #4). The 14-subject MCP server (Fleet primitive #6) is mounted alongside the agent for cross-subject tool calls.
